# Deal-history A/B — signal vs dumb vs grind (OPT vs ALT)

**Analysis only.** P&L from terminal `DEAL_PROFIT` + swap + commission — never telemetry `gross_pnl`.

**Data:** run `scripts/dump_deals_range.mq5`, copy CSV to `data/local/deals_dump_*.csv` (gitignored).

**Families:** signal (`2026`), dumb (`2126`), grind (`2226` — six OPT/ALT instances). Manual (`magic==0`) excluded from arm comparisons.

**Caveat:** this cycle had four defects, reloads, halts, and small N. **Indicative only — not conclusive.**


In [ ]:
import re
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from scipy.stats import mannwhitneyu
except ImportError:
    mannwhitneyu = None

ROOT = Path('D:/fxmatrix')
LOCAL = ROOT / 'data' / 'local'

# --- analysis window ---
ANALYSIS_FROM = '2026-09-03'
ANALYSIS_TO = None
PREFERRED_DUMP = 'deals_dump_20260910_0225.csv'  # None = auto-pick newest in data/local/

# --- v2 arms (retired mid-cycle; kept for continuity) ---
SIGNAL_MAGICS = {
    20260901, 20260902, 20260903, 20260904,
    20260911, 20260912, 20260913, 20260914,
    20260921, 20260922, 20260923, 20260924,
}
DUMB_MAGICS = {m + 1_000_000 for m in SIGNAL_MAGICS}

# --- grind family: prefix 2226; slot = last digit (1=OPT, 2=ALT) ---
GRIND_MAGICS = {
    22260101, 22260102,  # GBPUSD OPT / ALT
    22260201, 22260202,  # EURUSD OPT / ALT
    22260301, 22260302,  # EURGBP OPT / ALT
}
# Pair from magic middle digits (01/02/03) — authoritative; deal symbol is sanity-check only.
GRIND_PAIR_FROM_MAGIC = {
    22260101: 'GBPUSD', 22260102: 'GBPUSD',
    22260201: 'EURUSD', 22260202: 'EURUSD',
    22260301: 'EURGBP', 22260302: 'EURGBP',
}
GRIND_SLOT_FROM_MAGIC = {m: ('OPT' if m % 10 == 1 else 'ALT') for m in GRIND_MAGICS}

# Maker-economics cross-check: exit_pips × $0.10/pip − $0.06 commission (no spread)
GRIND_EXIT_PIPS = {
    'GBPUSD': {'OPT': 5, 'ALT': 7},
    'EURUSD': {'OPT': 7, 'ALT': 7},
    'EURGBP': {'OPT': 2, 'ALT': 5},
}
PIP_VALUE_USD = 0.10
COMMISSION_RT = 0.06

GRIND_COMMENT_RE = re.compile(
    r'^GRIND\|(?P<slot>OPT|ALT)\|(?P<side>L|S)\|L(?P<layer>\d+)\|(?P<role>ENT|EXT)',
    re.I,
)


In [ ]:
def find_dump() -> Path:
    if PREFERRED_DUMP:
        p = LOCAL / PREFERRED_DUMP
        if p.is_file():
            return p
    candidates = sorted(LOCAL.glob('deals_dump_*.csv'), key=os.path.getmtime, reverse=True)
    if candidates:
        return candidates[0]
    raise FileNotFoundError('No deals dump in data/local/')


def pip_size(symbol: str) -> float:
    return 0.01 if 'JPY' in symbol else 0.0001


def classify_arm(magic: int) -> str:
    if magic == 0:
        return 'manual'
    if magic in SIGNAL_MAGICS or 20260900 <= magic <= 20260999:
        return 'signal'
    if magic in DUMB_MAGICS or 21260900 <= magic <= 21260999:
        return 'dumb'
    if magic in GRIND_MAGICS or 22260000 <= magic <= 22269999:
        return 'grind'
    return 'other'


def side_from_v2_magic(magic: int) -> str:
    d = int(magic) % 10
    if d == 1:
        return 'long'
    if d == 2:
        return 'short'
    return 'exit/other'


def side_from_grind_comment(comment: str) -> str:
    m = GRIND_COMMENT_RE.match(str(comment))
    if not m:
        return '?'
    return 'long' if m.group('side').upper() == 'L' else 'short'


def classify_kind(row) -> str:
    c = str(row.get('comment', ''))
    entry = str(row.get('entry', '')).lower()

    gm = GRIND_COMMENT_RE.match(c)
    if gm:
        role = gm.group('role').upper()
        if role == 'ENT' and entry == 'in':
            return 'layer_open'
        if role == 'EXT' and entry == 'in':
            return 'exit_hedge'
        if entry in ('out', 'out_by'):
            return 'layer_close'
        return 'other'

    if entry == 'in' and (c.startswith('V2_L0') or c in ('V2_Add', 'V2_Reload')):
        return 'layer_open'
    if entry in ('out', 'out_by'):
        return 'layer_close'
    if c == 'V2_Exit' and entry == 'in':
        return 'exit_hedge'
    return 'other'


def grind_pair_from_open(open_magic: int, symbol: str) -> str:
    pair = GRIND_PAIR_FROM_MAGIC.get(int(open_magic))
    if pair and pair != symbol:
        pass  # magic encoding is authoritative; symbol mismatch is informational only
    return pair or symbol


def maker_prediction(pair: str, slot: str) -> float:
    return GRIND_EXIT_PIPS[pair][slot] * PIP_VALUE_USD - COMMISSION_RT


def mw_pvalue(a, b) -> float:
    if mannwhitneyu is None or len(a) < 2 or len(b) < 2:
        return float('nan')
    return float(mannwhitneyu(a, b, alternative='two-sided').pvalue)


def diff_se_band(a, b):
    if len(a) < 2 or len(b) < 2:
        return float('nan'), float('nan')
    diff = a.mean() - b.mean()
    se = np.sqrt(a.std(ddof=1) ** 2 / len(a) + b.std(ddof=1) ** 2 / len(b))
    return diff, 2.0 * se


## Section 1 — Load deal dump

In [ ]:
dump_path = find_dump()
print('Using:', dump_path)

raw = pd.read_csv(dump_path)
rename = {'deal_ticket': 'ticket'}
raw = raw.rename(columns={k: v for k, v in rename.items() if k in raw.columns})

for col in ('profit', 'swap', 'commission', 'volume', 'price', 'magic', 'position_id'):
    if col in raw.columns:
        raw[col] = pd.to_numeric(raw[col], errors='coerce')
if 'commission' not in raw.columns:
    raw['commission'] = 0.0

raw['time'] = pd.to_datetime(raw['time'])
raw['comment'] = raw['comment'].astype(str)
raw['entry'] = raw['entry'].astype(str).str.lower()
raw['symbol'] = raw['symbol'].astype(str)

print(f'{len(raw)} deals (full dump), {raw.time.min()} -> {raw.time.max()}')

deals = raw.copy()
if ANALYSIS_FROM:
    deals = deals[deals.time >= pd.Timestamp(ANALYSIS_FROM)]
if ANALYSIS_TO:
    deals = deals[deals.time <= pd.Timestamp(ANALYSIS_TO)]
print(f'{len(deals)} deals after window filter [{ANALYSIS_FROM} .. {ANALYSIS_TO or "end"}]')


## Section 2 — Classify arms, kinds, grind slot/pair

- **SIGNAL** prefix `2026`, **DUMB** prefix `2126`, **GRIND** prefix `2226`.
- Grind **slot** (OPT/ALT) from magic last digit: `1` = OPT, `2` = ALT.
- Grind **pair** from magic middle digits (`01` GBPUSD, `02` EURUSD, `03` EURGBP) — not from symbol.
- Parses both comment formats: `V2_*` and `GRIND|SLOT|SIDE|Lnn|ROLE` (`ENT` → layer_open, `EXT` → exit hedge).
- **Manual** = magic `0` (includes FTMO liquidation closes) — excluded from arm comparison.


In [ ]:
print('Unique magics in window:')
print(deals.groupby('magic').size().sort_index())

deals['arm'] = deals['magic'].astype(int).map(classify_arm)
deals['kind'] = deals.apply(classify_kind, axis=1)

print('\nArm × kind:')
print(deals.groupby(['arm', 'kind']).size())

unknown_magics = deals.loc[deals.arm == 'other', 'magic'].unique()
if len(unknown_magics):
    print('\nWARNING: unclassified magics:', sorted(unknown_magics))

other_kind = deals[deals.kind == 'other']
if other_kind.empty:
    print('\nkind=="other": empty ✓')
else:
    print(f'\nWARNING: {len(other_kind)} kind=="other" rows remain:')
    display(other_kind[['time', 'magic', 'arm', 'entry', 'comment', 'profit']].sort_values('time'))

manual = deals[deals.arm == 'manual']
print(f'\nManual (magic==0) deals: {len(manual)} rows')

grind_deals = deals[deals.magic.astype(int).isin(GRIND_MAGICS)]
print(f'\nGrind instances (by magic):')
print(grind_deals.groupby('magic').size().sort_index())


## Section 3 — Pair round-trip scalps by `position_id`

Round-trips are classified by **opening deal magic** (not the closing deal — FTMO closes use magic `0`).

Grind scalps close entirely via **CloseBy** (`DEAL_ENTRY_OUT_BY`); v2 arms are mixed OUT/OUT_BY.


In [ ]:
deals['net_pnl'] = deals['profit'].fillna(0) + deals['swap'].fillna(0) + deals['commission'].fillna(0)

opens = deals[deals.kind == 'layer_open'].copy()
open_meta = opens.groupby('position_id').agg(
    open_time=('time', 'min'),
    symbol=('symbol', 'first'),
    open_magic=('magic', 'first'),
    open_comment=('comment', 'first'),
).reset_index()

# Arm from OPENING magic — critical for FTMO liquidations (close magic = 0).
open_meta['arm'] = open_meta['open_magic'].astype(int).map(classify_arm)
open_meta['side'] = open_meta.apply(
    lambda r: side_from_grind_comment(r.open_comment)
    if int(r.open_magic) in GRIND_MAGICS
    else side_from_v2_magic(int(r.open_magic)),
    axis=1,
)
open_meta['slot'] = open_meta['open_magic'].map(lambda m: GRIND_SLOT_FROM_MAGIC.get(int(m), None))
open_meta['pair'] = open_meta.apply(
    lambda r: grind_pair_from_open(int(r.open_magic), r.symbol)
    if int(r.open_magic) in GRIND_MAGICS
    else r.symbol,
    axis=1,
)

pos_pnl = deals.groupby('position_id').agg(
    scalp_pnl=('net_pnl', 'sum'),
    last_time=('time', 'max'),
    n_deals=('ticket', 'count'),
).reset_index()

closed_pos = set(deals.loc[deals.entry.isin(['out', 'out_by']), 'position_id'].astype(int))

ftmo_closes = deals[
    (deals.arm == 'manual') & deals.comment.str.contains('CLOSED_BY_FTMO', case=False, na=False)
]
ftmo_pos = set(ftmo_closes['position_id'].astype(int))

scalps = open_meta.merge(pos_pnl, on='position_id', how='left')
scalps['completed'] = scalps['position_id'].isin(closed_pos)
scalps['is_ftmo'] = scalps['position_id'].isin(ftmo_pos)
scalps['hold_min'] = (scalps['last_time'] - scalps['open_time']).dt.total_seconds() / 60.0

# Grind pairing diagnostics
grind_opens = set(opens[opens.magic.astype(int).isin(GRIND_MAGICS)]['position_id'])
grind_deals_arm = deals[deals.magic.astype(int).isin(GRIND_MAGICS)]
print('=== GRIND PAIRING ===')
print(f'Grind layer opens: {len(grind_opens)}')
print(f'Paired (closed): {len(grind_opens & closed_pos)}')
print(f'Unmatched opens: {len(grind_opens - closed_pos)}')
print(f'Grind OUT_BY closes: {(grind_deals_arm.entry == "out_by").sum()}')
print(f'Grind OUT closes: {(grind_deals_arm.entry == "out").sum()}')

ftmo_scalps = scalps[(scalps.arm == 'grind') & scalps.is_ftmo]
print('\n=== FTMO LIQUIDATIONS (grind opens, magic-0 close) ===')
print(f'Count: {len(ftmo_scalps)}  Total P&L: ${ftmo_scalps.scalp_pnl.sum():+.2f}')
print('(Excluded from per-scalp statistics.)')

completed_scalps = scalps[scalps.completed & ~scalps.is_ftmo].copy()
print(f'\nCompleted scalps (all families, ex-FTMO): {len(completed_scalps)}')
print(completed_scalps.groupby('arm').size())


## Section 4 — P&L reconciliation gate

In [ ]:
account_realized = deals.net_pnl.sum()
parts = {
    arm: deals.loc[deals.arm == arm, 'net_pnl'].sum()
    for arm in ['signal', 'dumb', 'grind', 'manual', 'other']
}
check = sum(parts.values())
reconciled = abs(account_realized - check) < 0.01

print('P&L RECONCILIATION')
print('-' * 60)
print(f'Account realized (all deals):  ${account_realized:+.2f}')
for arm, pnl in parts.items():
    print(f'  {arm:8s}                     ${pnl:+.2f}')
print(f'Sum (signal+dumb+grind+manual+other): ${check:+.2f}')
print(f'\nTIE: {reconciled}  (delta ${account_realized - check:+.4f})')
if not reconciled:
    print('\n*** RECONCILIATION FAILED — do not treat statistics below as final ***')
else:
    print('\nReconciliation gate passed ✓')


## Section 5 — Family running dates & signal vs dumb (continuity)

Both v2 arms were retired mid-cycle; grind started later. Dates below are **actual deal activity** in the analysis window.


In [ ]:
if not reconciled:
    print('Skipping — reconciliation failed.')
else:
    print('=== FAMILY DATE RANGES ===')
    for arm in ['signal', 'dumb', 'grind']:
        sub = deals[deals.arm == arm]
        if sub.empty:
            print(f'{arm}: (no deals)')
        else:
            print(f'{arm}: {sub.time.min()} -> {sub.time.max()}')

    v2 = completed_scalps[completed_scalps.arm.isin(['signal', 'dumb'])].copy()
    print('\n=== SIGNAL vs DUMB (completed scalps, ex-FTMO) ===')
    for arm in ['signal', 'dumb']:
        s = v2[v2.arm == arm]
        hrs = max((s.open_time.max() - s.open_time.min()).total_seconds() / 3600, 1e-9)
        print(
            f'{arm:6s} N={len(s):3d}  total=${s.scalp_pnl.sum():+.2f}  '
            f'mean=${s.scalp_pnl.mean():+.3f}  turnover={len(s)/hrs:.1f} scalps/hr  '
            f'med_hold={s.hold_min.median():.1f} min'
        )

    sig = v2.loc[v2.arm == 'signal', 'scalp_pnl']
    dumb = v2.loc[v2.arm == 'dumb', 'scalp_pnl']
    diff, band = diff_se_band(sig, dumb)
    p = mw_pvalue(sig, dumb)
    print(f'\nMean diff (signal − dumb): ${diff:+.3f}  2×SE band: ±${band:.3f}  Mann-Whitney p={p:.3f}')
    print(v2.groupby(['arm', 'side']).size())


## Section 6 — OPT vs ALT (headline)

**Primary read: per-pair** — exit geometries differ (GBPUSD 5/5 vs 5/7, EURUSD 7/5 vs 7/7, EURGBP 3/2 vs 3/5). Pooled comparison shown for reference.

FTMO liquidations excluded. Small-N caveat applies throughout.


In [ ]:
if not reconciled:
    print('Skipping — reconciliation failed.')
else:
    grind = completed_scalps[completed_scalps.arm == 'grind'].copy()
    print('=== OPT vs ALT — per pair ===')
    for pair in ['GBPUSD', 'EURUSD', 'EURGBP']:
        sub = grind[grind.pair == pair]
        opt = sub.loc[sub.slot == 'OPT', 'scalp_pnl']
        alt = sub.loc[sub.slot == 'ALT', 'scalp_pnl']
        diff, band = diff_se_band(opt, alt)
        p = mw_pvalue(opt, alt)
        print(f'\n{pair}:')
        if len(opt):
            print(f'  OPT N={len(opt)} total=${opt.sum():+.2f} mean=${opt.mean():+.3f}')
        else:
            print('  OPT N=0')
        if len(alt):
            print(f'  ALT N={len(alt)} total=${alt.sum():+.2f} mean=${alt.mean():+.3f}')
        else:
            print('  ALT N=0')
        print(f'  OPT−ALT mean diff: ${diff:+.3f}  2×SE: ±${band:.3f}  p={p:.3f}')
        print(sub.groupby(['slot', 'side']).size())

    opt_all = grind.loc[grind.slot == 'OPT', 'scalp_pnl']
    alt_all = grind.loc[grind.slot == 'ALT', 'scalp_pnl']
    diff, band = diff_se_band(opt_all, alt_all)
    p = mw_pvalue(opt_all, alt_all)
    print('\n=== POOLED (all pairs) ===')
    print(f'OPT N={len(opt_all)} mean=${opt_all.mean():+.3f}  ALT N={len(alt_all)} mean=${alt_all.mean():+.3f}')
    print(f'OPT−ALT: ${diff:+.3f}  2×SE: ±${band:.3f}  Mann-Whitney p={p:.3f}')

    # turnover / hold per slot
    print('\n=== Turnover & hold (grind scalps) ===')
    for slot in ['OPT', 'ALT']:
        s = grind[grind.slot == slot]
        if s.empty:
            continue
        hrs = max((s.open_time.max() - s.open_time.min()).total_seconds() / 3600, 1e-9)
        print(
            f'{slot}: N={len(s)} turnover={len(s)/hrs:.2f}/hr  med_hold={s.hold_min.median():.1f} min'
        )


## Section 7 — Maker-economics cost cross-check (grind only)

In [ ]:
if not reconciled:
    print('Skipping — reconciliation failed.')
else:
    grind = completed_scalps[completed_scalps.arm == 'grind'].copy()
    print('Prediction: exit_pips × $0.10 − $0.06 commission (no spread)')
    print('-' * 55)
    for pair in GRIND_EXIT_PIPS:
        for slot in ['OPT', 'ALT']:
            sub = grind[(grind.pair == pair) & (grind.slot == slot)]
            if sub.empty:
                print(f'{pair} {slot}: no scalps in window')
                continue
            pred = maker_prediction(pair, slot)
            live = sub.scalp_pnl.mean()
            print(
                f'{pair} {slot}: live ${live:+.3f}  pred ${pred:+.3f}  '
                f'delta ${live - pred:+.3f}  N={len(sub)}'
            )


## Section 8 — Quantitative caveat

In [ ]:
print('=' * 72)
print('QUANTITATIVE CAVEAT')
print('=' * 72)
print(
    'This cycle was interrupted by four defects and multiple reloads. '
    'Counters reset on reload; several instances were halted for hours. '
    'Sample sizes are small (grind N≈40 completed scalps ex-FTMO). '
    'All comparisons below are INDICATIVE, not conclusive.'
)
if reconciled:
    g = completed_scalps[completed_scalps.arm == 'grind']
    print(f'\nGrind completed scalps (ex-FTMO): N={len(g)}')
    print(f'FTMO liquidations excluded: N={len(ftmo_scalps)}')
print('=' * 72)
